In [1]:
import os
import json
import random
import minsearch
import pandas as pd
import numpy as np
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI()

/Users/pauenpiang/afc-sushi-assistant/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Ingestion & Indexing

In [2]:
def load_csv(filename):
    for path in [filename, f"../data/{filename}", f"data/{filename}"]:
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f"Could not find {filename}")

df_safety = load_csv('food_safety.csv')
df_rolls = load_csv('OneRoll_updated.csv')

df = pd.concat([df_safety, df_rolls], ignore_index=True)
df = df.fillna('')

# Clean records explicitly to prevent vectorizer crashes from NaN/None values
documents = []
for idx, row in df.iterrows():
    doc = {col: ('' if pd.isna(row[col]) or str(row[col]).lower() in ['nan', 'none'] else str(row[col])) for col in df.columns}
    doc['id'] = idx
    documents.append(doc)

print(f"Loaded {len(df_safety)} rows from food_safety.csv and {len(df_rolls)} rows from OneRoll_updated.csv (Total: {len(documents)} documents)")

text_fields = [col for col in df.columns if col != 'id']
index = minsearch.Index(
    text_fields=text_fields,
    keyword_fields=[]
)
index.fit(documents)

Loaded 11 rows from food_safety.csv and 62 rows from OneRoll_updated.csv (Total: 73 documents)


## 2. RAG Flow Definition

In [3]:
def search(query, boost=None):
    if boost is None:
        boost = {}
    return index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

prompt_template = """You're a professional sushi assistant expert. Answer the QUESTION based on the CONTEXT from our food safety and sushi rolls database. Use only the facts from the CONTEXT when answering the QUESTION. If the answer cannot be found in the context, state that you don't have enough information.

QUESTION: {question}
CONTEXT:
{context}""".strip()

def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        doc_str = ", ".join([f"{k}: {v}" for k, v in doc.items() if k != 'id' and v])
        context += f"- {doc_str}\n\n"
    return prompt_template.format(question=query, context=context.strip())

def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def rag(query, model='gpt-4o-mini', boost=None):
    search_results = search(query, boost)
    prompt = build_prompt(query, search_results)
    return llm(prompt, model=model)

# Test query
question = 'What bacteria must be prevented during the preparation of sushi rice?'
answer = rag(question)
print(answer)

The context does not provide specific information about the bacteria that must be prevented during the preparation of sushi rice. Therefore, I don't have enough information to answer the question.


## 3. Retrieval Evaluation 

To get updated Generate question, run generate_questions.ipynb before running this code,

In [4]:
df_question = load_csv('../data/sushi-ground-truth-retrieval.csv')
ground_truth = df_question.to_dict(orient='records')
print(f"Loaded {len(ground_truth)} ground truth questions for evaluation.")

def hit_rate(relevance_total):
    cnt = sum(1 for line in relevance_total if True in line)
    return cnt / len(relevance_total) if relevance_total else 0

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank, is_relevant in enumerate(line):
            if is_relevant:
                total_score += 1 / (rank + 1)
    return total_score / len(relevance_total) if relevance_total else 0

def evaluate(ground_truth_data, search_function):
    relevance_total = []
    for q in tqdm(ground_truth_data, desc="Evaluating retrieval"):
        doc_id = int(q['id'])
        results = search_function(q)
        relevance = [int(d['id']) == doc_id for d in results]
        relevance_total.append(relevance)
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

metrics = evaluate(ground_truth, lambda q: search(q['question']))
print("Baseline Retrieval Evaluation Results:", metrics)

Loaded 365 ground truth questions for evaluation.


Evaluating retrieval: 100%|██████████| 365/365 [00:01<00:00, 305.61it/s]

Baseline Retrieval Evaluation Results: {'hit_rate': 0.9753424657534246, 'mrr': 0.6940791476407911}


## 4. RAG Quality Evaluation (LLM-as-a-Judge)

In [6]:
import re

prompt2_template = """You are an expert evaluator for a RAG system. Your task is to analyze the relevance of the generated answer to the given question. Based on the relevance of the generated answer, you will classify it as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Question: {question}
Generated Answer: {answer_llm}

Provide your evaluation in parsable JSON without using code blocks:
{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}} """.strip()

sample = df_question.sample(n=min(50, len(df_question)), random_state=1).to_dict(orient='records')
evaluations = []

for record in tqdm(sample, desc="Evaluating RAG quality"):
    question_text = record['question']
    answer_llm = rag(question_text, model='gpt-4o-mini')
    prompt = prompt2_template.format(question=question_text, answer_llm=answer_llm)
    evaluation_str = llm(prompt, model='gpt-4o-mini')
    try:
        # Automatically strip markdown code blocks if the LLM includes them
        cleaned_str = re.sub(r"^```(?:json)?\s*|\s*```$", "", evaluation_str.strip(), flags=re.IGNORECASE)
        evaluation = json.loads(cleaned_str)
    except Exception:
        evaluation = {"Relevance": "PARTLY_RELEVANT", "Explanation": "Failed to parse JSON evaluation."}
    evaluations.append((record, answer_llm, evaluation))

df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])
df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])
df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d.get('Relevance', ''))
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d.get('Explanation', ''))

print("\nRAG Evaluation Results Breakdown:")
print(df_eval['relevance'].value_counts(normalize=True))
df_eval.to_csv('../data/rag-eval-results.csv', index=False)

Evaluating RAG quality: 100%|██████████| 50/50 [01:23<00:00,  1.67s/it]


RAG Evaluation Results Breakdown:
relevance
RELEVANT           0.96
PARTLY_RELEVANT    0.04
Name: proportion, dtype: float64


# Display 3 examples for each relevance category

In [7]:
# Display 5 examples for each relevance category
for category in ['RELEVANT', 'PARTLY_RELEVANT', 'NON_RELEVANT']:
    print(f"\n{'='*20} CATEGORY: {category} {'='*20}")
    subset = df_eval[df_eval['relevance'] == category].head(3)
    
    if subset.empty:
        print(f"No examples found for {category}.")
        continue
        
    for idx, row in subset.iterrows():
        print(f"Question: {row['question']}")
        print(f"Answer:   {row['answer']}")
        print(f"Explanation: {row['explanation']}")
        print("-" * 60)


==================== CATEGORY: RELEVANT ====================
Question: How many pieces are there in a serving of the Hawaiian Roll?
Answer:   There are 10 pieces in a serving of the Hawaiian Roll (both versions).
Explanation: The generated answer directly addresses the question by providing the specific number of pieces in a serving of Hawaiian Roll, which is 10. This information is clear and directly related to the inquiry.
------------------------------------------------------------
Question: What are the main ingredients in the Crunchy CA Roll?
Answer:   The main ingredients in the Crunchy CA Roll are rice (or brown rice), seaweed, cucumber, crab stick, avocado, sesame seed, spicy sauce, sushi sauce, fried onion, ginger, wasabi, and baran.
Explanation: The generated answer lists all the main ingredients of the Crunchy CA Roll, directly addressing the question asked.
------------------------------------------------------------
Question: What is the recommended packing instruction fo

# Keyword Field Boosting

In [8]:
import random

# 1. Define the optimization function
def optimize_field_boosts(ground_truth_data, iterations=50):
    best_mrr = 0.0
    best_boost = {}
    
    # Get all text fields from your dataframe (excluding the 'id')
    text_fields = [col for col in df.columns if col != 'id']
    
    print(f"Starting Random Search optimization for {len(text_fields)} fields...")

    for i in range(iterations):
        # Generate random weights between 0.5 and 3.0 for every text field
        current_boost = {field: round(random.uniform(0.5, 3.0), 2) for field in text_fields}
        
        # Define search function with the current random boost weights
        def search_with_current_boost(q):
            return index.search(
                query=q['question'],
                filter_dict={},
                boost_dict=current_boost,
                num_results=10
            )
        
        # Evaluate MRR
        metrics = evaluate(ground_truth_data, search_with_current_boost)
        mrr = metrics['mrr']
        
        # Keep track of the best results
        if mrr > best_mrr:
            best_mrr = mrr
            best_boost = current_boost
            print(f"Iteration {i+1}: New Best MRR = {best_mrr:.4f} | Weights: {best_boost}")

    return best_boost, best_mrr

# 2. Run the optimization
best_boost, best_mrr = optimize_field_boosts(ground_truth)

print("\n--- Optimization Complete ---")
print("Best Boosting Parameters:")
print(json.dumps(best_boost, indent=4))

Starting Random Search optimization for 9 fields...


Evaluating retrieval: 100%|██████████| 365/365 [00:01<00:00, 311.11it/s]


Iteration 1: New Best MRR = 0.6833 | Weights: {'Item_Name': 1.39, 'Category': 1.7, 'Style': 0.74, 'Ingredients': 1.43, 'Assembly_Notes': 0.52, 'Packing_Instructions': 2.02, 'Piece_Count': 0.54, 'Raw_Cooked': 1.69, 'Rice_Type': 2.0}


Evaluating retrieval: 100%|██████████| 365/365 [00:01<00:00, 304.64it/s]


Iteration 3: New Best MRR = 0.6883 | Weights: {'Item_Name': 2.93, 'Category': 2.84, 'Style': 2.71, 'Ingredients': 1.12, 'Assembly_Notes': 1.1, 'Packing_Instructions': 1.95, 'Piece_Count': 0.76, 'Raw_Cooked': 2.34, 'Rice_Type': 1.31}


...

Iteration 20: New Best MRR = 0.7004 | Weights: {'Item_Name': 2.82, 'Category': 1.78, 'Style': 1.4, 'Ingredients': 1.11, 'Assembly_Notes': 2.98, 'Packing_Instructions': 2.71, 'Piece_Count': 2.32, 'Raw_Cooked': 0.82, 'Rice_Type': 1.19}


Evaluating retrieval: 100%|██████████| 365/365 [00:01<00:00, 316.66it/s]


Iteration 33: New Best MRR = 0.7026 | Weights: {'Item_Name': 1.99, 'Category': 1.52, 'Style': 1.85, 'Ingredients': 1.19, 'Assembly_Notes': 1.39, 'Packing_Instructions': 2.99, 'Piece_Count': 1.99, 'Raw_Cooked': 2.09, 'Rice_Type': 1.29}


Evaluating retrieval: 100%|██████████| 365/365 [00:01<00:00, 311.99it/s]


--- Optimization Complete ---
Best Boosting Parameters:
{
    "Item_Name": 1.99,
    "Category": 1.52,
    "Style": 1.85,
    "Ingredients": 1.19,
    "Assembly_Notes": 1.39,
    "Packing_Instructions": 2.99,
    "Piece_Count": 1.99,
    "Raw_Cooked": 2.09,
    "Rice_Type": 1.29
}


***After KeywordFieldBoosting, the LLM-Judge-Evaluation Results***

To get the updated llm-judge-eva results, first change the keyword boosting fields in rag.py.

In [31]:
# import re

# prompt2_template = """You are an expert evaluator for a RAG system. Your task is to analyze the relevance of the generated answer to the given question. Based on the relevance of the generated answer, you will classify it as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

# Question: {question}
# Generated Answer: {answer_llm}

# Provide your evaluation in parsable JSON without using code blocks:
# {{
#   "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
#   "Explanation": "[Provide a brief explanation for your evaluation]"
# }} """.strip()

# sample = df_question.sample(n=min(50, len(df_question)), random_state=1).to_dict(orient='records')
# evaluations = []

# for record in tqdm(sample, desc="Evaluating RAG quality"):
#     question_text = record['question']
#     answer_llm = rag(question_text, model='gpt-4o-mini')
#     prompt = prompt2_template.format(question=question_text, answer_llm=answer_llm)
#     evaluation_str = llm(prompt, model='gpt-4o-mini')
#     try:
#         # Automatically strip markdown code blocks if the LLM includes them
#         cleaned_str = re.sub(r"^```(?:json)?\s*|\s*```$", "", evaluation_str.strip(), flags=re.IGNORECASE)
#         evaluation = json.loads(cleaned_str)
#     except Exception:
#         evaluation = {"Relevance": "PARTLY_RELEVANT", "Explanation": "Failed to parse JSON evaluation."}
#     evaluations.append((record, answer_llm, evaluation))

# df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])
# df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
# df_eval['question'] = df_eval.record.apply(lambda d: d['question'])
# df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d.get('Relevance', ''))
# df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d.get('Explanation', ''))

# print("\nRAG Evaluation Results Breakdown:")
# print(df_eval['relevance'].value_counts(normalize=True))
# df_eval.to_csv('../data/rag-eval-results.csv', index=False)

In [ ]:
# # Display 3 examples for each relevance category
# for category in ['RELEVANT', 'PARTLY_RELEVANT', 'NON_RELEVANT']:
#     print(f"\n{'='*20} CATEGORY: {category} {'='*20}")
#     subset = df_eval[df_eval['relevance'] == category].head(3)
    
#     if subset.empty:
#         print(f"No examples found for {category}.")
#         continue
        
#     for idx, row in subset.iterrows():
#         print(f"Question: {row['question']}")
#         print(f"Answer:   {row['answer']}")
#         print(f"Explanation: {row['explanation']}")
#         print("-" * 60)